# 1. What you'll learn

        - build masks for real FashionMNIST foreground pixels
- trace skip-connection tensor shapes
- evaluate Dice and IoU rather than image accuracy

        **The one question this notebook answers:** How can an encoder-decoder predict one class per pixel while preserving fine boundaries?

# 2. Setup

This lesson keeps u-net segmentation small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader,Subset,Dataset
from torchvision import datasets,transforms

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Real cached FashionMNIST clothing images provide foreground shapes. Masks are deterministic annotations from non-background pixels, creating a small real-image segmentation task without another large download.

**Small example:** For five pixels, truth [1,1,0,0,1] and prediction [1,0,0,1,1] overlap on 2 pixels. Dice=2×2/(3+3)=2/3; IoU=2/(3+3-2)=1/2.

In [ ]:
torch.manual_seed(SEED);cache=Path.cwd().parent/"_shared_data";base=datasets.FashionMNIST(cache,train=True,download=False,transform=transforms.ToTensor())
class Segments(Dataset):
 def __init__(self,base,ids): self.base=base;self.ids=ids
 def __len__(self): return len(self.ids)
 def __getitem__(self,j):
  x,_=self.base[self.ids[j]];return x,(x>.12).float()
train_ds=Segments(base,list(range(1400)));test_ds=Segments(base,list(range(1400,1800)));print("Real image source: FashionMNIST; train/test:",len(train_ds),len(test_ds),"image/mask:",train_ds[0][0].shape,train_ds[0][1].shape)

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
pairs=[train_ds[i] for i in range(8)];images=torch.stack([p[0] for p in pairs]);masks=torch.stack([p[1] for p in pairs]);areas=torch.tensor([(train_ds[i][1].mean()) for i in range(len(train_ds))]);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack(images[:,0]),cmap="gray");axes[0].axis("off");axes[0].set_title("Real clothing images")
axes[1].imshow(np.hstack(masks[:,0]),cmap="gray");axes[1].axis("off");axes[1].set_title("Foreground masks");axes[2].hist(areas.numpy(),bins=25,color="#176b66");axes[2].set(title="Foreground fraction",xlabel="positive-pixel share");plt.tight_layout();plt.show()

**Takeaway:** Clothing boundaries vary by category and pose.

**Takeaway:** The mask labels every pixel, not just every image.

**Takeaway:** Background dominates many images, so raw pixel accuracy would reward empty predictions.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
gen=torch.Generator().manual_seed(SEED);train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,generator=gen,num_workers=0);test_loader=DataLoader(test_ds,batch_size=128,shuffle=False,num_workers=0);x,m=next(iter(train_loader));print("Batch image/mask:",x.shape,m.shape,"positive share:",round(m.mean().item(),3))

# 6. Train

        Training turns the assumptions behind u-net segmentation into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Encode local patterns while pooling spatial resolution.
2. Decode coarse features by upsampling.
3. Concatenate matching encoder maps through skip connections.
4. Predict one foreground logit per pixel and optimize binary cross-entropy.

In [ ]:
class TinyUNet(nn.Module):
 def __init__(self):
  super().__init__();self.e1=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.Conv2d(8,8,3,padding=1),nn.ReLU());self.down=nn.Sequential(nn.MaxPool2d(2),nn.Conv2d(8,16,3,padding=1),nn.ReLU());self.up=nn.ConvTranspose2d(16,8,2,stride=2);self.out=nn.Sequential(nn.Conv2d(16,8,3,padding=1),nn.ReLU(),nn.Conv2d(8,1,1))
 def forward(self,x): skip=self.e1(x);z=self.down(skip);return self.out(torch.cat([self.up(z),skip],1))
net=TinyUNet();opt=torch.optim.Adam(net.parameters(),lr=.003);losses=[]
for epoch in range(3):
 total=0
 for x,m in train_loader: opt.zero_grad();loss=nn.functional.binary_cross_entropy_with_logits(net(x),m);loss.backward();opt.step();total+=loss.item()*len(x)
 losses.append(total/len(train_ds));print(f"epoch {epoch+1}: pixel BCE={losses[-1]:.4f}")

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
def collect(dim=.0):
 ps=[];ys=[]
 with torch.no_grad():
  for x,m in test_loader: ps.append(torch.sigmoid(net(x*dim if dim else x)));ys.append(m)
 return torch.cat(ps),torch.cat(ys)
prob,truth=collect();pred=prob>.5;intersection=(pred*truth.bool()).sum((1,2,3)).float();dice=(2*intersection/(pred.sum((1,2,3))+truth.sum((1,2,3))+1e-7)).mean().item();iou=(intersection/((pred|truth.bool()).sum((1,2,3))+1e-7)).mean().item();empty_dice=0.0
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(range(1,4),losses,marker="o");axes[0].set_title("Pixel loss")
axes[1].imshow(np.hstack(truth[:8,0]),cmap="gray");axes[1].axis("off");axes[1].set_title("True masks");axes[2].imshow(np.hstack(pred[:8,0]),cmap="gray");axes[2].axis("off");axes[2].set_title(f"Predictions · Dice {dice:.3f}");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Dice={dice:.3f}; IoU={iou:.3f}; empty-mask Dice baseline={empty_dice:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Foreground appearance and intensity can shift; a model trained on one contrast may produce empty masks even while pixel accuracy looks high. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
dim_prob,dim_truth=collect(dim=.15);dim_pred=dim_prob>.5;inter=(dim_pred*dim_truth.bool()).sum((1,2,3)).float();dim_dice=(2*inter/(dim_pred.sum((1,2,3))+dim_truth.sum((1,2,3))+1e-7)).mean().item();print(f"Normal Dice={dice:.3f}; severe intensity-shift Dice={dim_dice:.3f}")

**Use this when…** you need dense pixel labels and boundaries matter.

        **Don't use this when…** only image-level labels exist, annotation quality is poor, or tiny targets vanish at the chosen resolution.

        ## Try this

        1. Remove the skip connection and inspect edges.
2. Use soft Dice loss instead of BCE.
3. Train masks at thresholds .05 and .3 and compare label sensitivity.